In [1]:
# ---------------------------------------------------------------
# Imports and Configuration
# ---------------------------------------------------------------
from pathlib import Path
import json
import gc
import itertools
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.metrics import classification_report

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

INPUT_PATH = Path("baseline_engineered.parquet")
FEATURE_LIST_PATH = Path("engineered_feature_cols.json")

# Fixed defaults, overridden per-stage as the search proceeds
WINDOW_LENGTH = 120
WINDOW_STRIDE = 30
MAX_WINDOWS_PER_CLASS = 15_000
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
MAX_EPOCHS = 30
PATIENCE = 5

Using device: cuda


In [2]:
# ---------------------------------------------------------------
# Load data, build split flags — identical to the Models notebook,
# so results stay comparable to the preliminary run
# ---------------------------------------------------------------
with open(FEATURE_LIST_PATH) as f:
    engineered_feature_cols = json.load(f)

needed_cols = list(set(
    engineered_feature_cols + ["fault_type", "Datetime", "source_file", "n_chillers_on"]
))
data = pd.read_parquet(INPUT_PATH, columns=needed_cols)

data["month"] = data["Datetime"].dt.month
data = data.sort_values(["source_file", "Datetime"]).reset_index(drop=True)

train_months, val_months, test_months = list(range(1, 9)), [9, 10], [11, 12]
data["split"] = np.select(
    [data["month"].isin(train_months), data["month"].isin(val_months), data["month"].isin(test_months)],
    ["train", "val", "test"], default="unused",
)
print(f"Loaded: {data.shape[0]:,} rows; {data['split'].value_counts().to_dict()}")

Loaded: 12,612,960 rows; {'train': 8396640, 'val': 2108160, 'test': 2108160}


In [3]:
# ---------------------------------------------------------------
# Window construction and dataset/scaler helpers — same as the
# Models notebook, wrapped as reusable functions since Stage 3
# needs to rebuild windows under different window_length/stride
# ---------------------------------------------------------------
def build_window_index(df, split_name, window_length, stride, max_per_class=None, random_state=RANDOM_STATE):
    windows = []
    split_df = df[df["split"] == split_name]
    for source_file, group in split_df.groupby("source_file", sort=False):
        group = group.sort_values("Datetime")
        n = len(group)
        if n < window_length:
            continue
        idx = group.index.to_numpy()
        label = group["fault_type"].iloc[0]
        for start in range(0, n - window_length + 1, stride):
            windows.append((idx[start:start + window_length], label))
    windows_df = pd.DataFrame(windows, columns=["row_idx", "fault_type"])
    if max_per_class is not None:
        sampled_parts = []
        for fault_type, group in windows_df.groupby("fault_type"):
            sampled_parts.append(group.sample(n=min(len(group), max_per_class), random_state=random_state))
        windows_df = pd.concat(sampled_parts, ignore_index=True)
    return windows_df.reset_index(drop=True)


class ChillerWindowDataset(Dataset):
    def __init__(self, df, windows_df, feature_cols, scaler, label_encoder):
        self.feature_matrix = scaler.transform(df[feature_cols].astype(np.float32)).astype(np.float32)
        self.labels = label_encoder.transform(windows_df["fault_type"].values)
        self.row_idx_list = windows_df["row_idx"].values
        self.pos_lookup = pd.Series(np.arange(len(df)), index=df.index)

    def __len__(self):
        return len(self.row_idx_list)

    def __getitem__(self, i):
        positions = self.pos_lookup.loc[self.row_idx_list[i]].values
        x = self.feature_matrix[positions]
        y = self.labels[i]
        return torch.from_numpy(x), torch.tensor(y, dtype=torch.long)


def build_loaders(window_length, stride, max_per_class, batch_size):
    """Rebuilds windows, scaler, datasets, and loaders for a given windowing configuration."""
    train_windows = build_window_index(data, "train", window_length, stride, max_per_class=max_per_class)
    val_windows = build_window_index(data, "val", window_length, stride, max_per_class=None)

    fit_sample_idx = data[data["split"] == "train"].sample(n=200_000, random_state=RANDOM_STATE).index
    scaler = RobustScaler()
    scaler.fit(data.loc[fit_sample_idx, engineered_feature_cols].astype(np.float32))

    label_encoder = LabelEncoder()
    label_encoder.fit(data["fault_type"])

    train_dataset = ChillerWindowDataset(data, train_windows, engineered_feature_cols, scaler, label_encoder)
    val_dataset = ChillerWindowDataset(data, val_windows, engineered_feature_cols, scaler, label_encoder)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    class_counts = train_windows["fault_type"].value_counts().reindex(label_encoder.classes_)
    class_weights = (len(train_windows) / (len(label_encoder.classes_) * class_counts)).values
    class_weights_t = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

    return train_loader, val_loader, label_encoder, class_weights_t, len(train_windows), len(val_windows)

In [4]:
# ---------------------------------------------------------------
# Model classes — identical to the Models notebook, now parameterized
# ---------------------------------------------------------------
class LSTMClassifier(nn.Module):
    def __init__(self, n_features, hidden_size=64, num_layers=1, n_classes=9, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size=n_features, hidden_size=hidden_size, num_layers=num_layers,
                             batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, n_classes)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return self.fc(self.dropout(h_n[-1]))


class CNN1DClassifier(nn.Module):
    def __init__(self, n_features, n_classes=9, channels=(64, 128), kernel_size=5, dropout=0.2):
        super().__init__()
        layers, in_ch = [], n_features
        for out_ch in channels:
            layers += [nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, padding=kernel_size // 2),
                       nn.ReLU(), nn.BatchNorm1d(out_ch)]
            in_ch = out_ch
        self.conv = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(channels[-1], n_classes)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.conv(x)
        x = self.pool(x).squeeze(-1)
        return self.fc(self.dropout(x))

In [5]:
# ---------------------------------------------------------------
# Training + evaluation helper — returns macro F1 on validation,
# the selection objective defined in Section 6.4, plus the full
# classification report and training history for logging
# ---------------------------------------------------------------
def train_and_score(model, train_loader, val_loader, class_weights_t, label_encoder,
                     lr=LEARNING_RATE, max_epochs=MAX_EPOCHS, patience=PATIENCE):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(weight=class_weights_t)

    best_val_loss = float("inf")
    epochs_no_improve = 0
    best_state = None

    for epoch in range(max_epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                val_loss += criterion(model(xb), yb).item() * xb.size(0)
        val_loss /= len(val_loader.dataset)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break

    model.load_state_dict(best_state)

    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            preds = model(xb.to(DEVICE)).argmax(dim=1).cpu().numpy()
            all_preds.append(preds)
            all_labels.append(yb.numpy())
    y_pred = label_encoder.inverse_transform(np.concatenate(all_preds))
    y_true = label_encoder.inverse_transform(np.concatenate(all_labels))
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)

    return report["macro avg"]["f1-score"], report, best_val_loss

In [6]:
# ---------------------------------------------------------------
# Stage 1 — architecture capacity. Fixed window/lr/batch at the
# preliminary run's values. ~6 trials per architecture,
# per the bounded-search budget in Section 6.4.
# ---------------------------------------------------------------
train_loader, val_loader, label_encoder, class_weights_t, n_train, n_val = build_loaders(
    WINDOW_LENGTH, WINDOW_STRIDE, MAX_WINDOWS_PER_CLASS, BATCH_SIZE
)
n_features = len(engineered_feature_cols)
n_classes = len(label_encoder.classes_)

stage1_results = []

lstm_grid = list(itertools.product([32, 64, 128], [1, 2]))  # (hidden_size, num_layers)
for hidden_size, num_layers in lstm_grid:
    model = LSTMClassifier(n_features, hidden_size=hidden_size, num_layers=num_layers, n_classes=n_classes)
    macro_f1, report, val_loss = train_and_score(model, train_loader, val_loader, class_weights_t, label_encoder)
    stage1_results.append({"arch": "lstm", "hidden_size": hidden_size, "num_layers": num_layers,
                            "macro_f1": macro_f1, "val_loss": val_loss})
    print(f"LSTM hidden={hidden_size} layers={num_layers}: macro_f1={macro_f1:.3f}, val_loss={val_loss:.4f}")
    del model
    torch.cuda.empty_cache(); gc.collect()

cnn_grid = list(itertools.product([(32, 64), (64, 128), (64, 128, 256)], [3, 5, 7]))
for channels, kernel_size in cnn_grid:
    model = CNN1DClassifier(n_features, n_classes=n_classes, channels=channels, kernel_size=kernel_size)
    macro_f1, report, val_loss = train_and_score(model, train_loader, val_loader, class_weights_t, label_encoder)
    stage1_results.append({"arch": "cnn", "channels": str(channels), "kernel_size": kernel_size,
                            "macro_f1": macro_f1, "val_loss": val_loss})
    print(f"CNN channels={channels} kernel={kernel_size}: macro_f1={macro_f1:.3f}, val_loss={val_loss:.4f}")
    del model
    torch.cuda.empty_cache(); gc.collect()

stage1_df = pd.DataFrame(stage1_results)
stage1_df.to_csv("stage1_results.csv", index=False)
print("\nStage 1 results:")
print(stage1_df.sort_values(["arch", "macro_f1"], ascending=[True, False]))

LSTM hidden=32 layers=1: macro_f1=0.479, val_loss=1.1961
LSTM hidden=32 layers=2: macro_f1=0.444, val_loss=1.2870
LSTM hidden=64 layers=1: macro_f1=0.496, val_loss=1.1674
LSTM hidden=64 layers=2: macro_f1=0.485, val_loss=1.0783
LSTM hidden=128 layers=1: macro_f1=0.433, val_loss=1.2487
LSTM hidden=128 layers=2: macro_f1=0.439, val_loss=1.2671
CNN channels=(32, 64) kernel=3: macro_f1=0.538, val_loss=1.0233
CNN channels=(32, 64) kernel=5: macro_f1=0.596, val_loss=0.8285
CNN channels=(32, 64) kernel=7: macro_f1=0.624, val_loss=0.7666
CNN channels=(64, 128) kernel=3: macro_f1=0.613, val_loss=0.8878
CNN channels=(64, 128) kernel=5: macro_f1=0.626, val_loss=0.7901
CNN channels=(64, 128) kernel=7: macro_f1=0.635, val_loss=0.7537
CNN channels=(64, 128, 256) kernel=3: macro_f1=0.638, val_loss=0.7454
CNN channels=(64, 128, 256) kernel=5: macro_f1=0.615, val_loss=0.7028
CNN channels=(64, 128, 256) kernel=7: macro_f1=0.571, val_loss=0.9266

Stage 1 results:
    arch  hidden_size  num_layers  macro_

In [7]:
# ---------------------------------------------------------------
# Select best Stage 1 configuration per architecture
# ---------------------------------------------------------------
best_lstm_s1 = stage1_df[stage1_df["arch"] == "lstm"].sort_values("macro_f1", ascending=False).iloc[0]
best_cnn_s1 = stage1_df[stage1_df["arch"] == "cnn"].sort_values("macro_f1", ascending=False).iloc[0]
print("Best LSTM (Stage 1):", dict(best_lstm_s1))
print("Best CNN (Stage 1):", dict(best_cnn_s1))

Best LSTM (Stage 1): {'arch': 'lstm', 'hidden_size': np.float64(64.0), 'num_layers': np.float64(1.0), 'macro_f1': np.float64(0.49572204819331517), 'val_loss': np.float64(1.167360544306388), 'channels': nan, 'kernel_size': np.float64(nan)}
Best CNN (Stage 1): {'arch': 'cnn', 'hidden_size': np.float64(nan), 'num_layers': np.float64(nan), 'macro_f1': np.float64(0.6383929852991197), 'val_loss': np.float64(0.7453891197110563), 'channels': '(64, 128, 256)', 'kernel_size': np.float64(3.0)}


In [8]:
# ---------------------------------------------------------------
# Stage 2 — optimization parameters, fixing Stage 1's best config.
# Specifically targets the LSTM's observed training instability
# via the learning-rate search.
# ---------------------------------------------------------------
stage2_results = []
lr_grid = [1e-4, 3e-4, 1e-3, 3e-3]
batch_grid = [128, 256, 512]

for lr, batch_size in itertools.product(lr_grid, batch_grid):
    train_loader_b, val_loader_b, label_encoder, class_weights_t, _, _ = build_loaders(
        WINDOW_LENGTH, WINDOW_STRIDE, MAX_WINDOWS_PER_CLASS, batch_size
    )

    model = LSTMClassifier(n_features, hidden_size=int(best_lstm_s1["hidden_size"]),
                            num_layers=int(best_lstm_s1["num_layers"]), n_classes=n_classes)
    macro_f1, _, val_loss = train_and_score(model, train_loader_b, val_loader_b, class_weights_t,
                                             label_encoder, lr=lr)
    stage2_results.append({"arch": "lstm", "lr": lr, "batch_size": batch_size,
                            "macro_f1": macro_f1, "val_loss": val_loss})
    print(f"LSTM lr={lr} batch={batch_size}: macro_f1={macro_f1:.3f}, val_loss={val_loss:.4f}")
    del model; torch.cuda.empty_cache(); gc.collect()

    model = CNN1DClassifier(n_features, n_classes=n_classes,
                             channels=eval(best_cnn_s1["channels"]), kernel_size=int(best_cnn_s1["kernel_size"]))
    macro_f1, _, val_loss = train_and_score(model, train_loader_b, val_loader_b, class_weights_t,
                                             label_encoder, lr=lr)
    stage2_results.append({"arch": "cnn", "lr": lr, "batch_size": batch_size,
                            "macro_f1": macro_f1, "val_loss": val_loss})
    print(f"CNN  lr={lr} batch={batch_size}: macro_f1={macro_f1:.3f}, val_loss={val_loss:.4f}")
    del model; torch.cuda.empty_cache(); gc.collect()

stage2_df = pd.DataFrame(stage2_results)
stage2_df.to_csv("stage2_results.csv", index=False)
print("\nStage 2 results:")
print(stage2_df.sort_values(["arch", "macro_f1"], ascending=[True, False]))

LSTM lr=0.0001 batch=128: macro_f1=0.446, val_loss=1.2713
CNN  lr=0.0001 batch=128: macro_f1=0.398, val_loss=1.4045
LSTM lr=0.0001 batch=256: macro_f1=0.415, val_loss=1.3476
CNN  lr=0.0001 batch=256: macro_f1=0.455, val_loss=1.2463
LSTM lr=0.0001 batch=512: macro_f1=0.359, val_loss=1.4731
CNN  lr=0.0001 batch=512: macro_f1=0.457, val_loss=1.2879
LSTM lr=0.0003 batch=128: macro_f1=0.444, val_loss=1.2853
CNN  lr=0.0003 batch=128: macro_f1=0.599, val_loss=0.8550
LSTM lr=0.0003 batch=256: macro_f1=0.359, val_loss=1.4429
CNN  lr=0.0003 batch=256: macro_f1=0.612, val_loss=0.9491
LSTM lr=0.0003 batch=512: macro_f1=0.444, val_loss=1.2631
CNN  lr=0.0003 batch=512: macro_f1=0.558, val_loss=1.0352
LSTM lr=0.001 batch=128: macro_f1=0.578, val_loss=1.0538
CNN  lr=0.001 batch=128: macro_f1=0.624, val_loss=0.9190
LSTM lr=0.001 batch=256: macro_f1=0.512, val_loss=1.1722
CNN  lr=0.001 batch=256: macro_f1=0.477, val_loss=1.0730
LSTM lr=0.001 batch=512: macro_f1=0.472, val_loss=1.2115
CNN  lr=0.001 batch

In [9]:
# ---------------------------------------------------------------
# Select best Stage 2 configuration per architecture
# ---------------------------------------------------------------
best_lstm_s2 = stage2_df[stage2_df["arch"] == "lstm"].sort_values("macro_f1", ascending=False).iloc[0]
best_cnn_s2 = stage2_df[stage2_df["arch"] == "cnn"].sort_values("macro_f1", ascending=False).iloc[0]
print("Best LSTM (Stage 2):", dict(best_lstm_s2))
print("Best CNN (Stage 2):", dict(best_cnn_s2))

Best LSTM (Stage 2): {'arch': 'lstm', 'lr': np.float64(0.003), 'batch_size': np.int64(256), 'macro_f1': np.float64(0.5796404800576633), 'val_loss': np.float64(0.9445466407790761)}
Best CNN (Stage 2): {'arch': 'cnn', 'lr': np.float64(0.001), 'batch_size': np.int64(128), 'macro_f1': np.float64(0.6236434118995432), 'val_loss': np.float64(0.9189752544882678)}


In [10]:
# ---------------------------------------------------------------
# Fast dataset: precompute ALL windows as a single float32 numpy
# array up front, rather than doing a pandas lookup per sample.
# __getitem__ becomes pure array slicing — no pandas overhead at all.
# ---------------------------------------------------------------
class FastChillerWindowDataset(Dataset):
    def __init__(self, df, windows_df, feature_cols, scaler, label_encoder):
        feature_matrix = scaler.transform(df[feature_cols].astype(np.float32)).astype(np.float32)
        pos_lookup = pd.Series(np.arange(len(df)), index=df.index)

        n_windows = len(windows_df)
        window_length = len(windows_df["row_idx"].iloc[0])
        n_features = feature_matrix.shape[1]

        # Precompute every window's data into one big array — this is the
        # one-time cost that replaces thousands of repeated per-sample lookups
        self.X = np.empty((n_windows, window_length, n_features), dtype=np.float32)
        for i, idx_arr in enumerate(windows_df["row_idx"]):
            positions = pos_lookup.loc[idx_arr].values
            self.X[i] = feature_matrix[positions]

        self.y = label_encoder.transform(windows_df["fault_type"].values)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return torch.from_numpy(self.X[i]), torch.tensor(self.y[i], dtype=torch.long)

In [11]:
# ---------------------------------------------------------------
# Stage 3 — SCOPED grid, given Stage 2's runtime. 2 values each
# instead of 3, bracketing the current 120-minute/30-stride choice
# rather than the full original 3x3 grid. This is a documented,
# compute-driven scope reduction (Section 6.4), not an oversight.
# ---------------------------------------------------------------
window_grid = [120, 240]   # dropped 60 — least likely to help given ACF still >0.88 at 60 min
stride_grid = [15, 30]     # dropped 60 — coarser stride mainly saves compute, unlikely to help accuracy

ARCHITECTURES_TO_SEARCH = ["lstm", "cnn"]   # change to ["cnn"] if needed

In [12]:
# ---------------------------------------------------------------
# Stage 3 — windowing parameters, using the fast dataset and the
# scoped grid above. Rebuilds are necessary here (window/stride
# genuinely changes), but each one now uses FastChillerWindowDataset
# instead of the slow per-sample pandas lookup.
# ---------------------------------------------------------------
stage3_results = []

for window_length, stride in itertools.product(window_grid, stride_grid):
    train_windows_w = build_window_index(data, "train", window_length, stride, max_per_class=MAX_WINDOWS_PER_CLASS)
    val_windows_w = build_window_index(data, "val", window_length, stride, max_per_class=None)

    fit_sample_idx = data[data["split"] == "train"].sample(n=200_000, random_state=RANDOM_STATE).index
    scaler_w = RobustScaler()
    scaler_w.fit(data.loc[fit_sample_idx, engineered_feature_cols].astype(np.float32))
    label_encoder = LabelEncoder()
    label_encoder.fit(data["fault_type"])

    train_dataset_w = FastChillerWindowDataset(data, train_windows_w, engineered_feature_cols, scaler_w, label_encoder)
    val_dataset_w = FastChillerWindowDataset(data, val_windows_w, engineered_feature_cols, scaler_w, label_encoder)

    class_counts = train_windows_w["fault_type"].value_counts().reindex(label_encoder.classes_)
    class_weights = (len(train_windows_w) / (len(label_encoder.classes_) * class_counts)).values
    class_weights_t = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

    if "lstm" in ARCHITECTURES_TO_SEARCH:
        train_loader_w = DataLoader(train_dataset_w, batch_size=int(best_lstm_s2["batch_size"]), shuffle=True, num_workers=0)
        val_loader_w = DataLoader(val_dataset_w, batch_size=int(best_lstm_s2["batch_size"]), shuffle=False, num_workers=0)
        model = LSTMClassifier(n_features, hidden_size=int(best_lstm_s1["hidden_size"]),
                                num_layers=int(best_lstm_s1["num_layers"]), n_classes=n_classes)
        macro_f1, _, val_loss = train_and_score(model, train_loader_w, val_loader_w, class_weights_t,
                                                 label_encoder, lr=best_lstm_s2["lr"])
        stage3_results.append({"arch": "lstm", "window_length": window_length, "stride": stride,
                                "macro_f1": macro_f1, "val_loss": val_loss,
                                "n_train": len(train_windows_w), "n_val": len(val_windows_w)})
        print(f"LSTM window={window_length} stride={stride}: macro_f1={macro_f1:.3f}")
        del model; torch.cuda.empty_cache(); gc.collect()

    if "cnn" in ARCHITECTURES_TO_SEARCH:
        train_loader_w = DataLoader(train_dataset_w, batch_size=int(best_cnn_s2["batch_size"]), shuffle=True, num_workers=0)
        val_loader_w = DataLoader(val_dataset_w, batch_size=int(best_cnn_s2["batch_size"]), shuffle=False, num_workers=0)
        model = CNN1DClassifier(n_features, n_classes=n_classes,
                                 channels=eval(best_cnn_s1["channels"]), kernel_size=int(best_cnn_s1["kernel_size"]))
        macro_f1, _, val_loss = train_and_score(model, train_loader_w, val_loader_w, class_weights_t,
                                                 label_encoder, lr=best_cnn_s2["lr"])
        stage3_results.append({"arch": "cnn", "window_length": window_length, "stride": stride,
                                "macro_f1": macro_f1, "val_loss": val_loss,
                                "n_train": len(train_windows_w), "n_val": len(val_windows_w)})
        print(f"CNN  window={window_length} stride={stride}: macro_f1={macro_f1:.3f}")
        del model; torch.cuda.empty_cache(); gc.collect()

    del train_windows_w, val_windows_w, train_dataset_w, val_dataset_w
    gc.collect()

stage3_df = pd.DataFrame(stage3_results)
stage3_df.to_csv("stage3_results.csv", index=False)
print("\nStage 3 results:")
print(stage3_df.sort_values(["arch", "macro_f1"], ascending=[True, False]))

LSTM window=120 stride=15: macro_f1=0.487
CNN  window=120 stride=15: macro_f1=0.638
LSTM window=120 stride=30: macro_f1=0.430
CNN  window=120 stride=30: macro_f1=0.612
LSTM window=240 stride=15: macro_f1=0.520
CNN  window=240 stride=15: macro_f1=0.676
LSTM window=240 stride=30: macro_f1=0.436
CNN  window=240 stride=30: macro_f1=0.672

Stage 3 results:
   arch  window_length  stride  macro_f1  val_loss  n_train   n_val
5   cnn            240      15  0.675595  0.704897   135000  140184
7   cnn            240      30  0.672464  0.713679   128310   70104
1   cnn            120      15  0.638475  0.803329   135000  140376
3   cnn            120      30  0.611639  0.894902   128318   70200
4  lstm            240      15  0.520028  1.110194   135000  140184
0  lstm            120      15  0.487203  1.234825   135000  140376
6  lstm            240      30  0.435586  1.318447   128310   70104
2  lstm            120      30  0.430113  1.237078   128318   70200


In [13]:
# ---------------------------------------------------------------
# Final selected configurations, consolidated — this is what
# Section 6.4's Model Configuration Table and Chapter VII should
# report, replacing the "pending" placeholders
# ---------------------------------------------------------------
best_lstm_s3 = stage3_df[stage3_df["arch"] == "lstm"].sort_values("macro_f1", ascending=False).iloc[0]
best_cnn_s3 = stage3_df[stage3_df["arch"] == "cnn"].sort_values("macro_f1", ascending=False).iloc[0]

final_configs = {
    "lstm": {
        "hidden_size": int(best_lstm_s1["hidden_size"]), "num_layers": int(best_lstm_s1["num_layers"]),
        "lr": float(best_lstm_s2["lr"]), "batch_size": int(best_lstm_s2["batch_size"]),
        "window_length": int(best_lstm_s3["window_length"]), "stride": int(best_lstm_s3["stride"]),
        "val_macro_f1": float(best_lstm_s3["macro_f1"]),
    },
    "cnn": {
        "channels": eval(best_cnn_s1["channels"]), "kernel_size": int(best_cnn_s1["kernel_size"]),
        "lr": float(best_cnn_s2["lr"]), "batch_size": int(best_cnn_s2["batch_size"]),
        "window_length": int(best_cnn_s3["window_length"]), "stride": int(best_cnn_s3["stride"]),
        "val_macro_f1": float(best_cnn_s3["macro_f1"]),
    },
}

with open("final_hyperparameters.json", "w") as f:
    json.dump(final_configs, f, indent=2)

print(json.dumps(final_configs, indent=2))

{
  "lstm": {
    "hidden_size": 64,
    "num_layers": 1,
    "lr": 0.003,
    "batch_size": 256,
    "window_length": 240,
    "stride": 15,
    "val_macro_f1": 0.5200283200975693
  },
  "cnn": {
    "channels": [
      64,
      128,
      256
    ],
    "kernel_size": 3,
    "lr": 0.001,
    "batch_size": 128,
    "window_length": 240,
    "stride": 15,
    "val_macro_f1": 0.6755945530110207
  }
}
